In [4]:
from pathlib import Path
import pandas as pd

# Project directories
PROJECT_ROOT = Path("/Users/jonathantran/Desktop/nba_economics")
STAGING_DIR = PROJECT_ROOT / "data" / "staging"
CANONICAL_DIR = PROJECT_ROOT / "data" / "canonical"

# Load the existing canonical tables
seasons = pd.read_csv(CANONICAL_DIR / "seasons.csv")
franchises = pd.read_csv(CANONICAL_DIR / "franchises.csv")
franchise_aliases = pd.read_csv(
    CANONICAL_DIR / "franchise_season_aliases.csv"
)
players = pd.read_csv(CANONICAL_DIR / "players.csv")
team_seasons = pd.read_csv(CANONICAL_DIR / "team_seasons.csv")
player_team_seasons = pd.read_csv(
    CANONICAL_DIR / "player_team_seasons.csv"
)

print("Canonical tables loaded successfully.")

# Locate S07 without assuming its exact filename or subfolder
s07 = pd.read_csv(
    STAGING_DIR / "player_salaries" / "s07_player_salaries.csv"
)

print("\nS07 shape:", s07.shape)
print("S07 columns:", s07.columns.tolist())

Canonical tables loaded successfully.

S07 shape: (12206, 7)
S07 columns: ['rank', 'player', 'team', 'salary', 'season', 'year', 'position']


In [2]:
# List all CSVs in the staging directory.
all_staging_files = sorted(STAGING_DIR.rglob("*.csv"))

print("Total staging CSVs:", len(all_staging_files))

for path in all_staging_files:
    print(path.relative_to(PROJECT_ROOT))

Total staging CSVs: 183
data/staging/CBA regime/s13_cba_regime_history.csv
data/staging/CBA regime/s13_cba_regime_map.csv
data/staging/NBA Draft/All_Drafts.csv
data/staging/NBA Draft/nba_draft_2001.csv
data/staging/NBA Draft/nba_draft_2002.csv
data/staging/NBA Draft/nba_draft_2003.csv
data/staging/NBA Draft/nba_draft_2004.csv
data/staging/NBA Draft/nba_draft_2005.csv
data/staging/NBA Draft/nba_draft_2006.csv
data/staging/NBA Draft/nba_draft_2007.csv
data/staging/NBA Draft/nba_draft_2008.csv
data/staging/NBA Draft/nba_draft_2009.csv
data/staging/NBA Draft/nba_draft_2010.csv
data/staging/NBA Draft/nba_draft_2011.csv
data/staging/NBA Draft/nba_draft_2012.csv
data/staging/NBA Draft/nba_draft_2013.csv
data/staging/NBA Draft/nba_draft_2014.csv
data/staging/NBA Draft/nba_draft_2015.csv
data/staging/NBA Draft/nba_draft_2016.csv
data/staging/NBA Draft/nba_draft_2017.csv
data/staging/NBA Draft/nba_draft_2018.csv
data/staging/NBA Draft/nba_draft_2019.csv
data/staging/NBA Draft/nba_draft_2020.csv


In [5]:
# Inspect the source before designing C07
print("S07 shape:", s07.shape)

print("\nColumn types:")
print(s07.dtypes)

print("\nMissing values:")
print(s07.isna().sum())

print("\nExact duplicate rows:", s07.duplicated().sum())

print("\nSample records:")
display(s07.head(10))

S07 shape: (12206, 7)

Column types:
rank        int64
player        str
team          str
salary      int64
season        str
year        int64
position      str
dtype: object

Missing values:
rank         0
player       0
team         0
salary       0
season       0
year         0
position    57
dtype: int64

Exact duplicate rows: 0

Sample records:


,rank,player,team,salary,season,year,position
0,1,Kevin Garnett,Minnesota Timberwolves,19600000,2000-2001,2001,PF
1,2,Shaquille O'Neal,Los Angeles Lakers,19285000,2000-2001,2001,C
2,3,Alonzo Mourning,Miami Heat,16879000,2000-2001,2001,C
3,4,Juwan Howard,Dallas Mavericks,16875000,2000-2001,2001,PF
4,5,Hakeem Olajuwon,Houston Rockets,16685000,2000-2001,2001,C
5,6,Karl Malone,Utah Jazz,15750000,2000-2001,2001,PF
6,7,Dikembe Mutombo,Philadelphia 76ers,14422000,2000-2001,2001,C
7,8,Patrick Ewing,Seattle SuperSonics,14000000,2000-2001,2001,C
8,9,David Robinson,San Antonio Spurs,13196000,2000-2001,2001,C
9,10,Scottie Pippen,Portland Trail Blazers,13151000,2000-2001,2001,SF


In [6]:
# 1. Check season coverage and formats
print("S07 season examples:")
print(s07["season"].unique()[:10])

print("\nSeason count:", s07["season"].nunique())
print("Year range:", s07["year"].min(), "-", s07["year"].max())

# 2. Inspect team identifiers
print("\nS07 team examples:")
print(sorted(s07["team"].unique())[:40])
print("Unique team identifiers:", s07["team"].nunique())

# 3. Test candidate grains
print("\nDuplicate player-season combinations:")
print(s07.duplicated(["player", "season"]).sum())

print("\nDuplicate player-team-season combinations:")
print(s07.duplicated(["player", "team", "season"]).sum())

# 4. Inspect player names in canonical C03
print("\nC03 player columns:")
print(players.columns.tolist())

print("\nC03 sample:")
display(players.head(5))

# 5. Inspect canonical season and franchise identifiers
print("\nC01 season columns:")
print(seasons.columns.tolist())

print("\nC02 franchise columns:")
print(franchises.columns.tolist())

print("\nC05 player-team-season columns:")
print(player_team_seasons.columns.tolist())

S07 season examples:
<StringArray>
['2000-2001', '2001-2002', '2002-2003', '2003-2004', '2004-2005', '2005-2006',
 '2006-2007', '2007-2008', '2008-2009', '2009-2010']
Length: 10, dtype: str

Season count: 26
Year range: 2001 - 2026

S07 team examples:
['ADELAIDE 36ers', 'Atlanta Hawks', 'Barcelona FC Barcelona', 'Beijing Ducks', 'Bilbao Basket', 'Bilbao Basket Bilbao Basket', 'Boston Celtics', 'Brooklyn Nets', 'Charlotte Bobcats', 'Charlotte Hornets', 'Chicago Bulls', 'Cleveland Cavaliers', 'Dallas Mavericks', 'Denver Nuggets', 'Detroit Pistons', 'Fenerbahce Ulker Fenerbahce Ulker', 'Flamengo Flamengo', 'Golden State Warriors', 'Guangzhou Loong-Lions', 'Houston Rockets', 'Indiana Pacers', 'LA Clippers', 'London Lions', 'Los Angeles Clippers', 'Los Angeles Lakers', 'Maccabi Haifa Maccabi Haifa', 'Madrid Real Madrid', 'Melbourne Melbourne United', 'Melbourne Pnx', 'Memphis Grizzlies', 'Miami Heat', 'Milan Olimpia', 'Milwaukee Bucks', 'Minnesota Timberwolves', 'NO/Oklahoma City  Hornets',

,player_id,player_name,country,college,draft_year,draft_round,draft_number
0,3,Grant Long,USA,Eastern Michigan,1988,2,33
1,15,Eric Piatkowski,USA,Nebraska,1994,1,15
2,21,Greg Anthony,USA,UNLV,1991,1,12
3,26,Luc Longley,Australia,New Mexico,1991,1,7
4,28,Doug West,USA,Villanova,1989,2,38



C01 season columns:
['season', 'Year', 'Salary Cap', 'Salary Cap Floor Pct', 'Salary Cap Floor', 'Luxury Tax', '1st Apron', '2nd Apron', 'BAE', 'Non-Taxpayer MLE', 'Taxpayer MLE', 'Team Room MLE', 'cba_id']

C02 franchise columns:
['franchise_id', 'current_team_name', 'current_abbreviation']

C05 player-team-season columns:
['player_id', 'franchise_id', 'season', 'age', 'height_inches', 'weight_lbs', 'games_played', 'points_total', 'rebounds_total', 'assists_total', 'wins', 'losses', 'win_pct', 'minutes_per_game', 'field_goals_made_per_game', 'field_goals_attempted_per_game', 'field_goal_pct', 'three_pointers_made_per_game', 'three_pointers_attempted_per_game', 'three_point_pct', 'free_throws_made_per_game', 'free_throws_attempted_per_game', 'free_throw_pct', 'offensive_rebounds_per_game', 'defensive_rebounds_per_game', 'rebounds_per_game', 'assists_per_game', 'turnovers_per_game', 'steals_per_game', 'blocks_per_game', 'personal_fouls_per_game', 'points_per_game', 'plus_minus_per_game

In [7]:
# 1. Inspect all S07 team names and their record counts
print("S07 TEAM COUNTS")
display(
    s07["team"]
    .value_counts()
    .rename_axis("team")
    .reset_index(name="records")
)

# 2. Inspect players with multiple salary records in one season
print("\nMULTIPLE SALARY RECORDS PER PLAYER-SEASON")

multi_salary = s07[
    s07.duplicated(["player", "season"], keep=False)
].sort_values(["season", "player", "team"])

display(
    multi_salary[
        ["player", "team", "salary", "season", "year"]
    ]
)

# 3. Inspect the canonical historical team-name bridge
print("\nCANONICAL FRANCHISE ALIAS COLUMNS")
print(franchise_aliases.columns.tolist())

print("\nSAMPLE HISTORICAL ALIASES")
display(franchise_aliases.head(10))

# 4. Check exact player-name matching potential
s07_names = set(s07["player"].str.strip())
c03_names = set(players["player_name"].str.strip())

matched_names = s07_names & c03_names
unmatched_names = s07_names - c03_names

print("\nPLAYER NAME MATCHING")
print("Unique S07 player names:", len(s07_names))
print("Exact matches to C03:", len(matched_names))
print("Unmatched names:", len(unmatched_names))

print("\nSAMPLE UNMATCHED NAMES")
print(sorted(unmatched_names)[:40])

S07 TEAM COUNTS


,team,records
0,Milwaukee Bucks,423
1,Philadelphia 76ers,422
2,Houston Rockets,421
3,Memphis Grizzlies,420
4,San Antonio Spurs,418
5,New York Knicks,415
6,Washington Wizards,415
7,Dallas Mavericks,413
8,Chicago Bulls,412
9,Toronto Raptors,407



MULTIPLE SALARY RECORDS PER PLAYER-SEASON


,player,team,salary,season,year
2757,Marcus Williams,LA Clippers,50254,2007-2008,2008
2593,Marcus Williams,New Jersey Nets,1180200,2007-2008,2008
3631,Marcus Williams,Denver Nuggets,825497,2009-2010,2010
3609,Marcus Williams,Memphis Grizzlies,825497,2009-2010,2010
5548,Chris Johnson,Memphis Grizzlies,55718,2012-2013,2013
5514,Chris Johnson,Minnesota Timberwolves,346781,2012-2013,2013
5557,Chris Wright,Dallas Mavericks,27859,2012-2013,2013
5465,Chris Wright,Oklahoma City Thunder,473604,2012-2013,2013
11169,KJ Martin,LA Clippers,1930681,2023-2024,2024
11168,KJ Martin,Philadelphia 76ers,1930681,2023-2024,2024



CANONICAL FRANCHISE ALIAS COLUMNS
['franchise_id', 'season', 'team_abbreviation', 'full_team_name', 'short_team_name']

SAMPLE HISTORICAL ALIASES


,franchise_id,season,team_abbreviation,full_team_name,short_team_name
0,1610612737,2000-2001,ATL,Atlanta Hawks,Hawks
1,1610612738,2000-2001,BOS,Boston Celtics,Celtics
2,1610612739,2000-2001,CLE,Cleveland Cavaliers,Cavaliers
3,1610612741,2000-2001,CHI,Chicago Bulls,Bulls
4,1610612742,2000-2001,DAL,Dallas Mavericks,Mavericks
5,1610612743,2000-2001,DEN,Denver Nuggets,Nuggets
6,1610612744,2000-2001,GSW,Golden State Warriors,Warriors
7,1610612745,2000-2001,HOU,Houston Rockets,Rockets
8,1610612746,2000-2001,LAC,Los Angeles Clippers,Clippers
9,1610612747,2000-2001,LAL,Los Angeles Lakers,Lakers



PLAYER NAME MATCHING
Unique S07 player names: 2220
Exact matches to C03: 2040
Unmatched names: 180

SAMPLE UNMATCHED NAMES
['A.J. Price', 'Adam Haluska', 'Alex Hamilton', 'Andrew Andrews', 'Anzejs Pasecniks', 'Beau Beech', 'Billy Garrett Jr.', 'Bo McCalebb', 'Bobby Portis', 'Bogdan Bogdanovic', "Boniface N'Dong", 'Brady Heslip', 'Brandon Ashley', 'Brandon Heath', 'Brandon Wallace', 'Brian Butch', 'Brian Chase', 'Brian Zoubek', 'Bryce Alford', 'C.J. Giles', 'CJ Wilcox', 'Caleb Homesley', 'Cameron Jones', 'Cameron Reynolds', 'Carleton Scott', 'Clarence Weatherspoon', 'Corey Hawkins', 'Curtis Jerrells', "D'Vauntes Smith-Rivera", 'D.J. Strawberry', 'D.J. White', "Da'Sean Butler", 'Dan Mcclintock', 'David Padgett', 'Dennis Schroder', 'Derek Cooke Jr.', 'Deron Washington', 'Devin Searcy', 'Diamon Simpson', 'Didier Ilunga-Mbenga']


In [8]:
import re
import html
import unicodedata

# Normalize names without changing the original source data.
def normalize_text(value):
    if pd.isna(value):
        return ""

    value = html.unescape(str(value))
    value = unicodedata.normalize("NFKD", value)
    value = "".join(
        char for char in value
        if not unicodedata.combining(char)
    )
    value = value.casefold()
    value = re.sub(r"[^a-z0-9]+", " ", value)

    return " ".join(value.split())


# Work on copies of our existing datasets.
salary_audit = s07.copy()
alias_audit = franchise_aliases.copy()
player_audit = players.copy()

# Create standardized matching keys.
salary_audit["team_key"] = salary_audit["team"].map(normalize_text)
salary_audit["player_key"] = salary_audit["player"].map(normalize_text)

alias_audit["team_key"] = (
    alias_audit["full_team_name"].map(normalize_text)
)
player_audit["player_key"] = (
    player_audit["player_name"].map(normalize_text)
)

# Handle known team-name variations.
team_corrections = {
    "la clippers": "los angeles clippers",
    "no oklahoma city hornets": "new orleans oklahoma city hornets",
}

salary_audit["team_key"] = (
    salary_audit["team_key"]
    .replace(team_corrections)
)

# Check for ambiguous lookup keys before merging.
ambiguous_teams = (
    alias_audit
    .groupby(["season", "team_key"])["franchise_id"]
    .nunique()
)
ambiguous_teams = ambiguous_teams[ambiguous_teams > 1]

ambiguous_players = (
    player_audit
    .groupby("player_key")["player_id"]
    .nunique()
)
ambiguous_players = ambiguous_players[ambiguous_players > 1]

print("Ambiguous team-season lookup keys:", len(ambiguous_teams))
print("Ambiguous player-name lookup keys:", len(ambiguous_players))

# Preserve only unique mappings.
team_lookup = (
    alias_audit[
        ["season", "team_key", "franchise_id"]
    ]
    .drop_duplicates()
)

player_lookup = (
    player_audit[
        ["player_key", "player_id"]
    ]
    .drop_duplicates()
)

# Remove ambiguous player names from automatic matching.
player_lookup = player_lookup[
    ~player_lookup["player_key"].isin(ambiguous_players.index)
]

# Exclude ambiguous team keys from automatic matching.
team_lookup = team_lookup.set_index(
    ["season", "team_key"]
)

team_lookup = team_lookup[
    ~team_lookup.index.isin(ambiguous_teams.index)
].reset_index()

# Match salary rows to canonical identifiers.
salary_audit = salary_audit.merge(
    team_lookup,
    on=["season", "team_key"],
    how="left",
    validate="many_to_one"
)

salary_audit = salary_audit.merge(
    player_lookup,
    on="player_key",
    how="left",
    validate="many_to_one"
)

# Summarize record-level matching.
total = len(salary_audit)

team_matched = salary_audit["franchise_id"].notna().sum()
player_matched = salary_audit["player_id"].notna().sum()

both_matched = (
    salary_audit["franchise_id"].notna()
    & salary_audit["player_id"].notna()
).sum()

print("\nS07 RECORD-LEVEL MATCHING")
print("-" * 45)
print("Total salary records:", total)
print(f"Team matched: {team_matched:,} ({team_matched / total:.1%})")
print(f"Player matched: {player_matched:,} ({player_matched / total:.1%})")
print(f"Both matched: {both_matched:,} ({both_matched / total:.1%})")

# Show teams that still require investigation.
print("\nUNMATCHED TEAM NAMES")
display(
    salary_audit.loc[
        salary_audit["franchise_id"].isna(), "team"
    ]
    .value_counts()
    .rename_axis("team")
    .reset_index(name="records")
)

# Show the most frequent unmatched player names.
print("\nTOP 30 UNMATCHED PLAYER NAMES")
display(
    salary_audit.loc[
        salary_audit["player_id"].isna(), "player"
    ]
    .value_counts()
    .head(30)
    .rename_axis("player")
    .reset_index(name="records")
)

Ambiguous team-season lookup keys: 0
Ambiguous player-name lookup keys: 10

S07 RECORD-LEVEL MATCHING
---------------------------------------------
Total salary records: 12206
Team matched: 12,075 (98.9%)
Player matched: 11,823 (96.9%)
Both matched: 11,751 (96.3%)

UNMATCHED TEAM NAMES


,team,records
0,null Unknown,45
1,New Orleans Pelicans,23
2,Brooklyn Nets,11
3,Charlotte Hornets,8
4,Madrid Real Madrid,5
5,Perth Perth,4
6,London Lions,4
7,Fenerbahce Ulker Fenerbahce Ulker,3
8,Barcelona FC Barcelona,3
9,Flamengo Flamengo,3



TOP 30 UNMATCHED PLAYER NAMES


,player,records
0,Nene Hilario,16
1,Mike James,12
2,Bobby Portis,11
3,Marcus Williams,9
4,Chris Johnson,9
5,Didier Ilunga-Mbenga,8
6,Dwayne Jones II,8
7,Chris Wright,7
8,D.J. White,6
9,Clarence Weatherspoon,5


In [9]:
# Identify the unmatched records for known NBA franchises.
nba_unmatched_names = [
    "New Orleans Pelicans",
    "Brooklyn Nets",
    "Charlotte Hornets",
    "Oklahoma City Thunder",
]

print("UNMATCHED NBA TEAM RECORDS BY SEASON")

nba_team_exceptions = salary_audit[
    salary_audit["franchise_id"].isna()
    & salary_audit["team"].isin(nba_unmatched_names)
]

display(
    nba_team_exceptions
    .groupby(["team", "season"])
    .size()
    .reset_index(name="records")
    .sort_values(["team", "season"])
)

# Examine ambiguous player names and their possible canonical IDs.
print("\nAMBIGUOUS CANONICAL PLAYER NAMES")

ambiguous_player_details = player_audit[
    player_audit["player_key"].isin(ambiguous_players.index)
][["player_key", "player_id", "player_name"]]

display(
    ambiguous_player_details.sort_values(
        ["player_key", "player_id"]
    )
)

# Check which ambiguous salary records have exactly one
# plausible player ID in C05 for the same season.
ambiguous_salary_rows = salary_audit[
    salary_audit["player_key"].isin(ambiguous_players.index)
].copy()

c05_candidates = player_team_seasons[
    ["player_id", "franchise_id", "season"]
].drop_duplicates()

ambiguous_candidates = ambiguous_salary_rows.merge(
    c05_candidates,
    on=["franchise_id", "season"],
    how="left",
    suffixes=("", "_candidate")
)

# Retain only candidates whose normalized canonical name
# matches the salary source name.
candidate_names = player_audit[
    ["player_id", "player_key"]
].rename(columns={
    "player_id": "player_id_candidate",
    "player_key": "candidate_player_key"
})

ambiguous_candidates = ambiguous_candidates.merge(
    candidate_names,
    on="player_id_candidate",
    how="left"
)

ambiguous_candidates = ambiguous_candidates[
    ambiguous_candidates["player_key"]
    == ambiguous_candidates["candidate_player_key"]
]

print("\nAMBIGUOUS SALARY RECORDS WITH C05 CANDIDATES")

display(
    ambiguous_candidates[
        [
            "player", "team", "season",
            "salary", "player_id_candidate"
        ]
    ].head(30)
)

UNMATCHED NBA TEAM RECORDS BY SEASON


,team,season,records
0,Brooklyn Nets,2005-2006,1
1,Brooklyn Nets,2006-2007,1
2,Brooklyn Nets,2007-2008,3
3,Brooklyn Nets,2009-2010,1
4,Brooklyn Nets,2010-2011,1
5,Brooklyn Nets,2011-2012,4
6,Charlotte Hornets,2009-2010,1
7,Charlotte Hornets,2010-2011,2
8,Charlotte Hornets,2011-2012,2
9,Charlotte Hornets,2012-2013,3



AMBIGUOUS CANONICAL PLAYER NAMES


,player_key,player_id,player_name
310,brandon williams,1585,Brandon Williams
2117,brandon williams,1630314,Brandon Williams
1177,chris johnson,202419,Chris Johnson
1326,chris johnson,203187,Chris Johnson
1244,chris wright,202874,Chris Wright
1330,chris wright,203203,Chris Wright
75,dee brown,244,Dee Brown
883,dee brown,200793,Dee Brown
190,glen rice,779,Glen Rice
1336,glen rice,203318,Glen Rice



AMBIGUOUS SALARY RECORDS WITH C05 CANDIDATES


,player,team,season,salary,player_id_candidate
7,Patrick Ewing,Seattle SuperSonics,2000-2001,14000000,121.0
17,Glen Rice,New York Knicks,2000-2001,7578000,779.0
32,Dee Brown,Orlando Magic,2000-2001,548000,244.0
46,Glen Rice,Houston Rockets,2001-2002,8800000,779.0
69,Patrick Ewing,Orlando Magic,2001-2002,2118000,121.0
84,Mike James,Miami Heat,2001-2002,237000,2229.0
93,Glen Rice,Houston Rockets,2002-2003,9200000,779.0
111,Glen Rice,LA Clippers,2003-2004,9600000,779.0
131,Mike James,Toronto Raptors,2005-2006,3410000,2229.0
143,Mike James,Minnesota Timberwolves,2006-2007,5215000,2229.0


In [10]:
# STEP 1: Recover historical franchise-name mismatches
# using the current-name-to-franchise-ID relationship.

current_team_lookup = franchises[
    ["franchise_id", "current_team_name"]
].copy()

current_team_lookup["team_key"] = (
    current_team_lookup["current_team_name"].map(normalize_text)
)

# Check that current team names uniquely identify franchises.
assert not current_team_lookup["team_key"].duplicated().any()

current_team_lookup = current_team_lookup[
    ["team_key", "franchise_id"]
].rename(columns={"franchise_id": "current_franchise_id"})

salary_audit = salary_audit.merge(
    current_team_lookup,
    on="team_key",
    how="left",
    validate="many_to_one"
)

# Fill missing historical franchise IDs where the current-name
# mapping identifies a franchise.
salary_audit["franchise_id"] = (
    salary_audit["franchise_id"]
    .fillna(salary_audit["current_franchise_id"])
)

salary_audit = salary_audit.drop(columns="current_franchise_id")


# STEP 2: Resolve ambiguous player names with C05 context.

# Build the canonical player-team-season lookup.
c05_player_lookup = player_team_seasons[
    ["player_id", "franchise_id", "season"]
].drop_duplicates()

# Attach the canonical normalized player name.
c05_player_lookup = c05_player_lookup.merge(
    player_audit[["player_id", "player_key"]],
    on="player_id",
    how="left",
    validate="many_to_one"
)

# Retain only one candidate per unique combination.
c05_player_lookup = c05_player_lookup[
    ["player_key", "franchise_id", "season", "player_id"]
].drop_duplicates()

# Only accept keys with exactly one candidate player ID.
candidate_counts = (
    c05_player_lookup
    .groupby(["player_key", "franchise_id", "season"])["player_id"]
    .nunique()
    .reset_index(name="candidate_count")
)

unique_candidates = c05_player_lookup.merge(
    candidate_counts,
    on=["player_key", "franchise_id", "season"],
    how="inner"
)

unique_candidates = unique_candidates[
    unique_candidates["candidate_count"] == 1
].drop(columns="candidate_count")

unique_candidates = unique_candidates.rename(
    columns={"player_id": "c05_player_id"}
)

salary_audit = salary_audit.merge(
    unique_candidates,
    on=["player_key", "franchise_id", "season"],
    how="left",
    validate="many_to_one"
)

# Only fill player IDs that were previously unmatched.
salary_audit["player_id"] = (
    salary_audit["player_id"]
    .fillna(salary_audit["c05_player_id"])
)

salary_audit = salary_audit.drop(columns="c05_player_id")


# STEP 3: Report the updated match rates.

total = len(salary_audit)

team_matched = salary_audit["franchise_id"].notna().sum()
player_matched = salary_audit["player_id"].notna().sum()

both_matched = (
    salary_audit["franchise_id"].notna()
    & salary_audit["player_id"].notna()
).sum()

print("UPDATED C07 MATCHING RESULTS")
print("-" * 45)

print(f"Total records: {total:,}")
print(f"Team matched: {team_matched:,} ({team_matched / total:.1%})")
print(f"Player matched: {player_matched:,} ({player_matched / total:.1%})")
print(f"Both matched: {both_matched:,} ({both_matched / total:.1%})")

print("\nUnmatched NBA-team names:")
display(
    salary_audit.loc[
        salary_audit["franchise_id"].isna(), "team"
    ].value_counts().head(15)
)

print("\nRemaining unmatched player records:")
print(salary_audit["player_id"].isna().sum())

print("\nDuplicate canonical player-franchise-season keys:")
print(
    salary_audit.dropna(
        subset=["player_id", "franchise_id"]
    ).duplicated(
        ["player_id", "franchise_id", "season"]
    ).sum()
)

UPDATED C07 MATCHING RESULTS
---------------------------------------------
Total records: 12,206
Team matched: 12,118 (99.3%)
Player matched: 11,858 (97.1%)
Both matched: 11,826 (96.9%)

Unmatched NBA-team names:


team
null Unknown                         45
Madrid Real Madrid                    5
Perth Perth                           4
London Lions                          4
Fenerbahce Ulker Fenerbahce Ulker     3
Barcelona FC Barcelona                3
Flamengo Flamengo                     3
Guangzhou Loong-Lions                 3
Melbourne Melbourne United            3
Maccabi Haifa Maccabi Haifa           3
Milan Olimpia                         2
New Zealand Breakers                  2
Melbourne Pnx                         2
Bilbao Basket Bilbao Basket           1
Bilbao Basket                         1
Name: count, dtype: int64


Remaining unmatched player records:
348

Duplicate canonical player-franchise-season keys:
0


In [11]:
# Build C07 from records with both canonical identifiers.
matched_mask = (
    salary_audit["player_id"].notna()
    & salary_audit["franchise_id"].notna()
)

c07 = salary_audit.loc[
    matched_mask,
    [
        "player_id",
        "franchise_id",
        "season",
        "salary",
        "player",
        "team",
    ]
].copy()

c07 = c07.rename(
    columns={
        "player": "source_player_name",
        "team": "source_team_name",
    }
)

# Restore integer IDs after pandas merges introduced nullable values.
c07["player_id"] = c07["player_id"].astype("int64")
c07["franchise_id"] = c07["franchise_id"].astype("int64")

# Keep unresolved records in a separate audit dataset.
c07_unresolved = salary_audit.loc[
    ~matched_mask,
    [
        "player",
        "team",
        "season",
        "salary",
        "player_id",
        "franchise_id",
    ]
].copy()

# Validate uniqueness and essential values.
assert not c07.duplicated(
    ["player_id", "franchise_id", "season"]
).any(), "Duplicate C07 canonical keys detected."

assert c07["salary"].notna().all(), "Missing salary values detected."
assert (c07["salary"] >= 0).all(), "Negative salary values detected."

# Validate canonical identifiers.
assert c07["player_id"].isin(
    players["player_id"]
).all(), "Unknown player IDs detected."

assert c07["franchise_id"].isin(
    franchises["franchise_id"]
).all(), "Unknown franchise IDs detected."

assert c07["season"].isin(
    seasons["season"]
).all(), "Unknown seasons detected."

# Validate the actual franchise-season combinations against C04.
valid_team_seasons = team_seasons[
    ["franchise_id", "season"]
].drop_duplicates()

team_season_check = c07[
    ["franchise_id", "season"]
].merge(
    valid_team_seasons,
    on=["franchise_id", "season"],
    how="left",
    indicator=True,
    validate="many_to_one",
)

assert (
    team_season_check["_merge"] == "both"
).all(), "C07 contains franchise-season combinations absent from C04."

# Sort for a reproducible export.
c07 = c07.sort_values(
    ["season", "franchise_id", "player_id"]
).reset_index(drop=True)

# Report before writing.
print("C07 VALIDATION PASSED")
print("-" * 45)
print("Canonical salary rows:", len(c07))
print("Unresolved source rows:", len(c07_unresolved))
print("Seasons represented:", c07["season"].nunique())
print("Unique players:", c07["player_id"].nunique())
print("Unique franchises:", c07["franchise_id"].nunique())

# Export C07 and its audit file.
c07.to_csv(
    CANONICAL_DIR / "player_salaries.csv",
    index=False,
)

c07_unresolved.to_csv(
    CANONICAL_DIR / "player_salaries_unresolved.csv",
    index=False,
)

print("\nFiles exported successfully.")

C07 VALIDATION PASSED
---------------------------------------------
Canonical salary rows: 11826
Unresolved source rows: 380
Seasons represented: 26
Unique players: 2068
Unique franchises: 30

Files exported successfully.
